In [ ]:
import datasets
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
import numpy as np
import torchvision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [51]:
train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(10),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = Imagenette(root = './data', split = 'train', download = True, transform = train_transforms)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers = 7)

In [52]:
validation_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

validation_dataset = Imagenette(root = './data', split = 'val', download = True, transform = validation_transforms)
validation_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 7)

Попробуем в лоб запустить модель, обученную на Imagenet1k, на датасете Imagenette, и протестировать её по метрике F1.

In [71]:
# маппинг классов
imagenette_classes = [label[0] for label in validation_dataset.classes] + ['BAD_CLASS']

imagenet_to_imagenette = dict()
imagenet_to_imagenette[0] = 0
imagenet_to_imagenette[217] = 1
imagenet_to_imagenette[482] = 2
imagenet_to_imagenette[491] = 3
imagenet_to_imagenette[497] = 4
imagenet_to_imagenette[566] = 5
imagenet_to_imagenette[569] = 6
imagenet_to_imagenette[571] = 7
imagenet_to_imagenette[574] = 8
imagenet_to_imagenette[701] = 9

In [72]:
from tqdm import tqdm

def ValidateModel(model, device, data_loader):
  model.eval()
  true_labels = torch.tensor([]).to(DEVICE)
  predicted_labels = torch.tensor([]).to(DEVICE)

  with torch.no_grad():
    for images, labels in tqdm(data_loader):
      images = images.to(device)
      labels = labels.to(device)
      
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)
  return true_labels, predicted_labels

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from torchmetrics.classification import MulticlassF1Score

print(f"Model Resnet50: ")
resnet_model = resnet50(weights = ResNet50_Weights.IMAGENET1K_V2)
resnet_model = resnet_model.to(DEVICE)

resnet_model.eval()

true_data, predicted_data = ValidateModel(resnet_model, device = DEVICE, data_loader = validation_loader)
mapped_predicted = []
for value in predicted_data:
  if (value.item() in imagenet_to_imagenette.keys()):
    mapped_predicted.append(imagenet_to_imagenette[int(value)])
  else:
    mapped_predicted.append(10)
mapped_predicted = torch.tensor(mapped_predicted).to(DEVICE)

Model Resnet50: 


100%|██████████| 123/123 [00:23<00:00,  5.21it/s]


In [116]:
from sklearn.metrics import f1_score

print(f1_score(true_data.cpu().numpy(), mapped_predicted.cpu().numpy(), average = 'macro'))

0.861515543641725


Попробуем изменить модель: Отрежем её последний слой с 1000 нейронами, а на его место добавим новый слой на 10 нейронов, которые будут обучены на классификацию imagenette.

In [41]:
from torchvision.models import resnet50, ResNet50_Weights

modified_model = resnet50(weights = ResNet50_Weights.IMAGENET1K_V2)

for param in modified_model.parameters():
  param.requires_grad = False

last_layer = modified_model.fc.in_features
modified_model.fc = torch.nn.Linear(last_layer, 10)


In [ ]:
# Обучение модели на imagenette
import torch.optim as optim
from tqdm import tqdm

epochs_count = 5

optimizer = optim.Adam(modified_model.fc.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
train_losses = []
validation_accuracies = []

# Лучшие параметры
best_acc = 0
best_model_wts = None

modified_model = modified_model.to(DEVICE)

# Обучение
for epoch in range(1, epochs_count + 1):
  print(f"Epoch: {epoch} / {epochs_count}")
  modified_model.train()

  curr_loss = 0
  curr_corrects = 0

  for inputs, labels in tqdm(train_loader):
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)
    optimizer.zero_grad()
    with torch.no_grad():
      outputs = modified_model(inputs)
      _, preds = torch.max(outputs, 1)
      loss = criterion(outputs, labels)

      loss.backward()

      optimizer.step()

    curr_loss += loss.item() * inputs.size(0)
    curr_corrects += torch.sum(preds == labels.data)
  epoch_loss = curr_loss / len(train_loader.dataset)
  epoch_acc = curr_corrects.double() / len(train_loader.dataset)

  train_losses.append(epoch_loss)

  # Валидация
  modified_model.eval()
  val_corrects = 0
  for inputs, labels in tqdm(validation_loader):
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.set_grad_enabled(False):
      outputs = modified_model(inputs)
      _, preds = torch.max(outputs, 1)

    val_corrects += torch.sum(preds == labels.data)

  val_acc = val_corrects.double() / len(validation_loader.dataset)
  validation_accuracies.append(val_acc.item())

  print(f"Train Loss: {epoch_loss}, Train Acc: {epoch_acc}")
  print(f"Val Acc: {val_acc}")

  if val_acc > best_acc:
    best_acc = val_acc
    best_model_wts = modified_model.state_dict().copy()

modified_model.load_state_dict(best_model_wts)

Epoch: 1 / 5


100%|██████████| 123/123 [00:23<00:00,  5.16it/s]


Train Loss: 0.1379579846821084, Train Acc: 0.9562783820889217
Val Acc: 0.9956687898089172
Epoch: 2 / 5


100%|██████████| 123/123 [00:24<00:00,  5.12it/s]


Train Loss: 0.12233500084667148, Train Acc: 0.960925124089133
Val Acc: 0.9954140127388534
Epoch: 3 / 5


...

100%|██████████| 123/123 [00:23<00:00,  5.13it/s]


Train Loss: 0.12515715187324403, Train Acc: 0.9613475551800612
Val Acc: 0.9951592356687897
Epoch: 5 / 5


100%|██████████| 123/123 [00:24<00:00,  5.12it/s]

Train Loss: 0.11560523789897302, Train Acc: 0.9620868095891858
Val Acc: 0.9951592356687897


<All keys matched successfully>

In [45]:
torch.save({
    'model_state_dict': modified_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'modified_resnet50.pth')

In [119]:
print("Modified Model Resnet50:")
modified_model = modified_model.to(DEVICE)

modified_model.eval()

true_data, predicted_data = ValidateModel(modified_model, device = DEVICE, data_loader = validation_loader)
print(f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro'))

Modified Model Resnet50:


  0%|          | 0/123 [00:00<?, ?it/s]

100%|██████████| 123/123 [00:23<00:00,  5.13it/s]

0.9951466510618175
